## 1. 데이터 셋팅
### 1-1 분석환경 설정

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('fivethirtyeight')

import warnings
warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.4f}'.format

from pmdarima import auto_arima

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.tools import diff
from statsmodels.tools.eval_measures import rmse
from statsmodels.tsa.stattools import adfuller
import itertools


In [2]:
file_path ='C:/Users/82109/Downloads/상권분석데이터/'
df= pd.read_csv(file_path + '0727_Seoul_preprocessed.csv',encoding = 'euc-kr', index_col=0)

### 1-2. 아드풀러 함수 정의
- 시계열 데이터가 정상성이 있는지 혹은 트렌드나 계절성 때문에 차분이 필요한 상태인지 판별

In [3]:
#모델 확인
def adf_test(series,title=''):
    """
    Pass in a time series and an optional title, returns an ADF report
    """
    print(f'아드풀러 검정 결과(ADF Test): {title}')
    result = adfuller(series.dropna(),autolag='AIC') # .dropna() handles differenced data

    labels = ['ADF 통계량','p-value','사용된 시차','관측치 수']
    out = pd.Series(result[0:4],index=labels)

    for key,val in result[4].items():
        out[f'임계값 critical value ({key})']=val

    print(out.to_string())

    if result[1] <= 0.05:
        print("\n✨ [판정] p-value가 0.05 이하이므로 귀무가설을 기각합니다.")
        print("➡️ 이 데이터는 '정상성(Stationary)'을 만족합니다. (그냥 분석 가능)")
    else:
        print("\n⚠️ [판정] p-value가 0.05보다 크므로 귀무가설을 기각할 수 없습니다.")
        print("➡️ 이 데이터는 '비정상성(Non-stationary)'입니다. (차분/빼기 작업 필요!)")

## 2. 데이터 전처리 및 탐색
### 2-1. 분석 대상 데이터 추출 (신촌동, 커피-음료 업종)

In [4]:
df_coffee = df[(df['행정동']== '신촌동')&(df['업종명'] == '커피-음료')]
df_coffee['기준년분기'] = df_coffee['기준_년_코드'].astype(str) + "-" + df_coffee['기준_분기_코드'].astype(str)
df_coffee= df_coffee.groupby(['기준년분기'])['주중_매출_금액', '주말_매출_금액'].sum().reset_index()
df_coffee['총매출'] = df_coffee['주중_매출_금액'] + df_coffee['주말_매출_금액']
df_coffee = df_coffee.drop(['주중_매출_금액', '주말_매출_금액'], inplace=False, axis=1)

### 2-2. 시계열 정상성 검증(adf 검정)
- p-value가 0.05보다 크므로 차분이 필요함 

In [5]:
adf_test(df_coffee['총매출'], title='신촌 커피-음료 총매출 데이터')

아드풀러 검정 결과(ADF Test): 신촌 커피-음료 총매출 데이터
ADF 통계량                    -2.4958
p-value                     0.1165
사용된 시차                      8.0000
관측치 수                      15.0000
임계값 critical value (1%)    -3.9644
임계값 critical value (5%)    -3.0849
임계값 critical value (10%)   -2.6818

⚠️ [판정] p-value가 0.05보다 크므로 귀무가설을 기각할 수 없습니다.
➡️ 이 데이터는 '비정상성(Non-stationary)'입니다. (차분/빼기 작업 필요!)


## 3. 시계열 모델링 및 예측
- **auto arima**: 정보 손실을 최소화하는 최적의 파라미터 조합(AIC최소)자동 탐색
- **SARIMA:** 분기별(4분기 주기)로 반복되는 매출의 명확한 계절성 패턴을 포착하기 위해 적용
- **홀트 윈터스 모델:** 최근 매출, 추세, 주기성의 가중치를 조절하는 전통적 지수평활법 모델
- **Prophet:** 시계열을 [추세+ 계절성 + 이벤트]의 합으로 분해하여 변동성을 직관적 계산하는 모델 
- **성능검증 지표:** RMSE를 실제 매출 평균으로 나눈 값 (평균 매출 대비 오차율)

### 3-1. auto arima 성능 검증

탐색 결과 ARIMA(0,0,1)(0,1,1)[4]가 선정이 되었으며,통계적 검정 수행
- **모델 유의성 파악:** p-value가 0.023으로 통계적 유의수준 만족함?????
- 데이터 분할: 2017년 ~ 2021년 (학습 20개 분기) / 2022년 1~4분기 (검증 4개 분기)

In [6]:
# 데이터를 train(~21년 4분기), test(22년 1~4분기)로 분리합니다.

train = df_coffee.iloc[:20]
test = df_coffee.iloc[20:]

# train 데이터로 autoarima를 실행합니다. 
step_model = auto_arima(
    train['총매출'], 
    start_p=0, max_p=3, 
    start_q=0, max_q=3,
    d=None,
    seasonal=True, 
    m=4,
    start_P=0, max_P=2, 
    start_Q=0, max_Q=2,
    D=None,           
    trace=True,
    suppress_warnings=True
)

print(step_model.summary())

Performing stepwise search to minimize aic
 ARIMA(0,0,0)(0,1,0)[4] intercept   : AIC=735.417, Time=0.01 sec
 ARIMA(1,0,0)(1,1,0)[4] intercept   : AIC=743.812, Time=0.03 sec
 ARIMA(0,0,1)(0,1,1)[4] intercept   : AIC=732.361, Time=0.03 sec
 ARIMA(0,0,0)(0,1,0)[4]             : AIC=743.352, Time=0.00 sec
 ARIMA(0,0,1)(0,1,0)[4] intercept   : AIC=733.631, Time=0.02 sec
 ARIMA(0,0,1)(1,1,1)[4] intercept   : AIC=734.349, Time=0.04 sec
 ARIMA(0,0,1)(0,1,2)[4] intercept   : AIC=733.054, Time=0.04 sec
 ARIMA(0,0,1)(1,1,0)[4] intercept   : AIC=732.519, Time=0.02 sec
 ARIMA(0,0,1)(1,1,2)[4] intercept   : AIC=734.682, Time=0.07 sec
 ARIMA(0,0,0)(0,1,1)[4] intercept   : AIC=732.792, Time=0.02 sec
 ARIMA(1,0,1)(0,1,1)[4] intercept   : AIC=735.878, Time=0.06 sec
 ARIMA(0,0,2)(0,1,1)[4] intercept   : AIC=733.327, Time=0.06 sec
 ARIMA(1,0,0)(0,1,1)[4] intercept   : AIC=743.945, Time=0.04 sec
 ARIMA(1,0,2)(0,1,1)[4] intercept   : AIC=733.661, Time=0.11 sec
 ARIMA(0,0,1)(0,1,1)[4]             : AIC=742.3

In [7]:
best_order = (0, 0, 1)
best_seasonal_order = (0, 1, 1, 4)

print("============ [Step 1] auto_arima 지표 성능 검증 ============")

# 2. 검증용 모델 학습
val_model = SARIMAX(
    train['총매출'].values, 
    order=best_order, 
    seasonal_order=best_seasonal_order
)
val_results = val_model.fit(disp=False)

# 3. 2022년 1~4분기 예측 수행
start = len(train)
end = len(train) + len(test) - 1
val_predictions = val_results.predict(start=start, end=end)

# 4. 검증 결과 화면 출력 및 오차(RMSE) 계산
pred_val_df = pd.DataFrame({
    '기준년분기': test.index,
    '실제_총매출': test['총매출'].values,
    '예측_총매출': val_predictions
})

pred_val_df['오차율(%)'] = (abs(pred_val_df['실제_총매출'] - pred_val_df['예측_총매출']) / pred_val_df['실제_총매출']) * 100
print(pred_val_df.to_string(index=False))
# 최종 평균 오차율 (MAPE)
mape = pred_val_df['오차율(%)'].mean()
print(f"\n📉 평균 분기별 예측 오차율(MAPE): {mape:.2f}%")

============ [Step 1] auto_arima 지표 성능 검증 ============
 기준년분기      실제_총매출          예측_총매출  오차율(%)
    20  6499152739 5784211482.0667 11.0005
    21  9352509875 7443863919.6195 20.4078
    22  9566330395 6738421736.9232 29.5611
    23 10313798137 7154613372.8707 30.6307

📉 평균 분기별 예측 오차율(MAPE): 22.90%


#### 📊 auto arima 결과
- 데이터 분할: 2017년~ 2021년 (학습 데이터), 22년 (시험용 데이터)
- 평균 분기별 예측 오차율(MAPE): **22.90%**
> 인사이트: 코로나를 겪으며 보수적으로 예측을 하지만, 실제 하반기 매출 회복 속도가 빨라서 하반기로 갈수록 오차가 크게 발생한 것으로 분석됨

### 3-2. SARIMA 모델 + 매출 로그화
학습용 데이터를 한 분기 늘려 정확도 측정한 결과 오차율이 5.6% 줄어들었습니다. 
- 데이터 분할: 2017년~ 2022년 1분기 (학습 데이터), 22년 2~4분기 (시험용 데이터)
- 분석 결과(평균 매출 대비 오차율): **44.31%**
- 인사이트: 아직도 코로나를 겪으며 보수적으로 예측을 하지만, 실제 데이터엔 매출 회복 속도가 빨라서 오차가 큰 상황임 

In [8]:
print("============ [조정 Step 1] auto_arima 지표 + 데이터 갯수 늘려 성능 검증 ============")
df_coffee['총매출_log'] = np.log1p(df_coffee['총매출'])

train = df_coffee.iloc[:20] 
test = df_coffee.iloc[20:]

# train 데이터로 autoarima를 실행합니다. 
step_model = auto_arima(
    train['총매출_log'], 
    start_p=0, max_p=3, 
    start_q=0, max_q=3,
    d=None,
    seasonal=True, 
    m=4,
    start_P=0, max_P=2, 
    start_Q=0, max_Q=2,
    D=None,           
    trace=True,
    suppress_warnings=True
)

============ [조정 Step 1] auto_arima 지표 + 데이터 갯수 늘려 성능 검증 ============
Performing stepwise search to minimize aic
 ARIMA(0,0,0)(0,1,0)[4] intercept   : AIC=-1.417, Time=0.02 sec
 ARIMA(1,0,0)(1,1,0)[4] intercept   : AIC=-11.425, Time=0.09 sec
 ARIMA(0,0,1)(0,1,1)[4] intercept   : AIC=inf, Time=0.10 sec
 ARIMA(0,0,0)(0,1,0)[4]             : AIC=6.231, Time=0.02 sec
 ARIMA(1,0,0)(0,1,0)[4] intercept   : AIC=-12.371, Time=0.03 sec
 ARIMA(1,0,0)(0,1,1)[4] intercept   : AIC=inf, Time=0.10 sec
 ARIMA(1,0,0)(1,1,1)[4] intercept   : AIC=inf, Time=0.17 sec
 ARIMA(2,0,0)(0,1,0)[4] intercept   : AIC=-11.486, Time=0.04 sec
 ARIMA(1,0,1)(0,1,0)[4] intercept   : AIC=-11.379, Time=0.07 sec
 ARIMA(0,0,1)(0,1,0)[4] intercept   : AIC=inf, Time=0.06 sec
 ARIMA(2,0,1)(0,1,0)[4] intercept   : AIC=-13.503, Time=0.10 sec
 ARIMA(2,0,1)(1,1,0)[4] intercept   : AIC=-11.856, Time=0.20 sec
 ARIMA(2,0,1)(0,1,1)[4] intercept   : AIC=-15.338, Time=0.22 sec
 ARIMA(2,0,1)(1,1,1)[4] intercept   : AIC=-13.171, Time=0.19 

In [9]:
best_order = (2, 0, 1)
best_seasonal_order = (0, 1, 1, 4)

# 2. 검증용 모델 학습
val_model = SARIMAX(
    train['총매출_log'].values, 
    order=best_order, 
    seasonal_order=best_seasonal_order
)
val_results = val_model.fit(disp=False)


# 3. 2022년 1~4분기 예측 수행
start = len(train)
end = len(train) + len(test) - 1
val_pred_log = val_results.predict(start=start, end=end)
val_predictions = np.expm1(val_pred_log)

# 4. 검증 결과 화면 출력 및 오차(RMSE) 계산
pred_val_df = pd.DataFrame({
    '기준년분기': test.index,
    '실제_총매출': test['총매출'].values,
    '예측_총매출': val_predictions
})

val_results = val_model.fit(disp=False)


pred_val_df['오차율(%)'] = (abs(pred_val_df['실제_총매출'] - pred_val_df['예측_총매출']) / pred_val_df['실제_총매출']) * 100
print(pred_val_df.to_string(index=False))
# 최종 평균 오차율 (MAPE)
mape = pred_val_df['오차율(%)'].mean()
print(f"\n📉 평균 분기별 예측 오차율(MAPE): {mape:.2f}%")

 기준년분기      실제_총매출          예측_총매출  오차율(%)
    20  6499152739 5917720896.4916  8.9463
    21  9352509875 7035620677.4229 24.7729
    22  9566330395 6688497492.6752 30.0829
    23 10313798137 6647961432.4522 35.5430

📉 평균 분기별 예측 오차율(MAPE): 24.84%


### 3-4. SARIMA 모델 분석 결과
학습용 데이터를 한 분기 늘리고, 로그 변환과 파라미터 수동 튜닝을하여 정확도 측정한 결과, 오차율이 20.6% 줄어들었습니다. 
- 데이터 분할: 2017년~ 2022년 1분기 (학습 데이터), 22년 2~4분기 (시험용 데이터)
- 로그 변환: 매출 단위 변화를 줄여서 오차를 감소시킴
- 파라미터 수동 튜닝: 전 분기와 작년 동분기를 더 고려하게 조율 (p=1, P=1)  
- 분석 결과(평균 매출 대비 오차율): **35.17%**
- 인사이트: 코로나 이후 보수적인 예측치가 개선되었으나, 실제 매출 회복 속도가 더 큼

### 3-5. 홀트-윈터스 (Holt-Winters) 모델 분석 결과
홀트-윈터스 모델을 이용하여 예측한 결과 SARIMA 모델의 예측보다 오차율이 18.87% 개선되었습니다.
- **모델 핵심 설정 (Trend='add', Seasonal='mul'):**
    - **더해지는 추세(Additive):** 시간이 흐름에 따라 상권의 전반적인 매출 체급이 일정 수준씩 지속 선형 성장한다고 가정.
    - **곱해지는 계절성(Multiplicative):** 대학가 상권 특성상 매출 규모가 커질수록 방학/개강 시즌에 따른 분기별 매출 변동 폭(진폭)도 비례하여 확대되는 비즈니스 특성을 반영.
- **데이터 분할:** 2017년 ~ 2022년 1분기(학습 데이터) / 2022년 2분기 ~ 4분기(검증 데이터)
- **분석 결과 (평균 매출 대비 오차율):** **28.53%**
- **인사이트:** SARIMA 모델보다 오차율이 낮게 나타남. 복잡한 통계적 파라미터 조합보다, 최근 매출 트렌드와 직관적인 분기 가중치를 반영하는 지수평활법이 본 상권의 2022년 변동성을 더 안정적으로 예측함을 확인함

In [10]:
from statsmodels.tsa.api import ExponentialSmoothing
# 1. 안전하게 데이터 분리
train = df_coffee.iloc[:20]
test = df_coffee.iloc[20:]

# 2. 모델 정의 및 자동 파라미터 최적화 (.fit()에 옵션을 주어 내부 최적화 실행)
# 데이터의 성격에 맞춰 추세는 'add'(더하기), 계절성은 'add'와 'mul' 중 선택 가능합니다.
hw_model = ExponentialSmoothing(
    train['총매출'].astype(float), 
    trend= 'mul', 
    seasonal='mul', # 💡 'mul'보다 'add'가 데이터가 적을 때 훨씬 안정적입니다.
    seasonal_periods=4,
    damped_trend=True
)

# 🔥 내부 최적화 엔진 작동 (알파, 베타, 감마 파라미터를 자동으로 탐색)
hw_results = hw_model.fit(optimized=True, use_brute=True)

# 3. test 데이터 개수만큼 정확하게 예측 수행
hw_predictions = hw_results.forecast(steps=len(test))

# 4. 날짜 컬럼 자동 매핑으로 개수 에러 완벽 차단
# 만약 '기준년분기'가 컬럼이면 test['기준년분기'].values를, 인덱스면 test.index를 씁니다.
time_index = test['기준년분기'].values if '기준년분기' in test.columns else test.index

pred_val_df = pd.DataFrame({
    '기준년분기': time_index,
    '실제_총매출': test['총매출'].values,
    '예측_총매출': hw_predictions.values  # 계측값 개수 일치 보장
})

# 5. 오차율(MAPE) 계산 및 출력
pred_val_df['오차율(%)'] = (abs(pred_val_df['실제_총매출'] - pred_val_df['예측_총매출']) / pred_val_df['실제_총매출']) * 100
print(pred_val_df.to_string(index=False))

# 최종 평균 오차율 (MAPE)
mape = pred_val_df['오차율(%)'].mean()
print(f"\n📉 평균 분기별 예측 오차율(MAPE): {mape:.2f}%")

 기준년분기      실제_총매출          예측_총매출  오차율(%)
2022-1  6499152739 5742271562.8753 11.6458
2022-2  9352509875 6972360039.5790 25.4493
2022-3  9566330395 6735224147.6960 29.5945
2022-4 10313798137 6734521460.2623 34.7038

📉 평균 분기별 예측 오차율(MAPE): 25.35%


c:\Users\82109\anaconda3\lib\site-packages\statsmodels\tsa\holtwinters\model.py:915: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


### 3-5 Prophet 모델 분석 결과
Prophet 모델을 이용하여 예측한 결과 4.16%의 오차율을 기록하며, 다른 예측 모델보다 훨씬 정교한 예측 결과를 냄
- **모델 핵심 설정:**
    - holidays: 트렌드 계산에서 20년~21년을 예측하지 않아서 왜곡을 줄임
    - 시즌성: 연간 반복되는 상권 특성(방학 및 개강 시즌)만 반영하도록 최적화
- **데이터 분할:** 2017년 ~ 2022년 1분기(학습 데이터) / 2022년 2분기 ~ 4분기(검증 데이터)
- **분석 결과 (평균 매출 대비 오차율):** **4.16%**
- **인사이트:** 테스트한 모델 중 가장 정교하고 압도적인 예측 성능을 기록함. 전통적 통계 모델은 코로나 변동성을 처리하기 어려웠으나, Prophet은 요인을 독립적으로 분해할 수 있어서 상권 분석에 활용하기 좋음

In [12]:
!pip install prophet

In [ ]:
from prophet import Prophet

print("============ [Prophet 모델] 테스트 및 검증 ============")

# 2. Prophet 전용 데이터프레임 만들기
df_prophet = pd.DataFrame()
# 분기 텍스트를 파이썬이 인식하는 날짜 데이터로 깔끔하게 변환
df_prophet['ds'] = pd.to_datetime(df_coffee['기준년분기'].str.replace('-', 'Q'))
df_prophet['y'] = df_coffee['총매출']

# 3. 데이터 분리 (20개 / 4개)
train_p = df_prophet.iloc[:20]
test_p = df_prophet.iloc[20:]

# 4. 코로나 기간을 이벤트(휴일)로 등록하여 무시하게 만듭니다.
# (사장님의 이 아이디어는 진짜 최고입니다 👍)
lockdowns = pd.DataFrame({
  'holiday': 'corona',
  'ds': pd.to_datetime(['2020-03-31', '2020-06-30', '2020-09-30', '2020-12-31',
                        '2021-03-31', '2021-06-30', '2021-09-30', '2021-12-31']),
  'lower_window': 0,
  'upper_window': 0,
})

# 5. 모델 정의 및 학습
model_p = Prophet(
    holidays=lockdowns,
    yearly_seasonality=True,   # 분기 데이터이므로 연간 패턴 학습 필수
    weekly_seasonality=False, 
    daily_seasonality=False
)
model_p.fit(train_p)

# 6. ★핵심 수정: 미래 날짜를 새로 만들지 말고, test 데이터의 ds를 그대로 붙여서 가져옵니다★
# 이렇게 해야 날짜 꼬임 현상(인덱스 밀림)을 100% 방어할 수 있습니다.
future = df_prophet[['ds']] # 전체 날짜(24개)를 그대로 전달
forecast = model_p.predict(future)

# 7. 예측 결과 추출 (test 데이터와 일치하는 마지막 4개 분기만 쏙 뽑기)
pred_val = forecast['yhat'].iloc[20:].values

pred_val_df = pd.DataFrame({
    '기준년분기': df_coffee['기준년분기'].iloc[20:].tolist(),
    '실제_총매출': test_p['y'].values,
    '예측_총매출': pred_val
})

# 8. 앞선 모델들과 공정한 비교를 위해 MAPE 오차율 계산으로 통일!
pred_val_df['오차율(%)'] = (abs(pred_val_df['실제_총매출'] - pred_val_df['예측_총매출']) / pred_val_df['실제_총매출']) * 100
print(pred_val_df.to_string(index=False))

# 최종 평균 오차율 (MAPE)
mape = pred_val_df['오차율(%)'].mean()
print(f"\n📉 평균 분기별 예측 오차율(MAPE): {mape:.2f}%")

============ [Prophet 모델] 테스트 및 검증 ============


AttributeError: 'Prophet' object has no attribute 'stan_backend'

In [ ]:
# 전체 데이터로 Prophet 재학습 및 2023년 진짜 미래 예측
final_model_p = Prophet(holidays=lockdowns, yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
final_model_p.fit(df_prophet)

# 2023년 4개 분기 미래 날짜 생성 (2023년 3월, 6월, 9월, 12월 말일 기준)
future_2023 = final_model_p.make_future_dataframe(periods=4, freq='Q')

# 예측하기
forecast_2023 = final_model_p.predict(future_2023)

# 2023년 결과만 이쁘게 잘라내기
real_forecast = forecast_2023[['ds', 'yhat']].iloc[24:].reset_index(drop=True)
real_forecast.columns = ['기준년분기', '2023년_예측_총매출']

# 날짜 모양을 원래 보기 좋던 분기 형태로 변환 (예: 2023-1)
real_forecast['기준년분기'] = ['2023-1', '2023-2', '2023-3', '2023-4']

print("============ Prophet 기반 2023년 미래 매출 예측 ============")
print(real_forecast.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# 그래프 크기 설정
plt.figure(figsize=(12, 6))

# 1. 과거 실제 데이터 그리기 (17년 1분기 ~ 22년 4분기)
plt.plot(df_coffee['기준년분기'], df_coffee['총매출'], marker='o', label='과거 실제 매출', color='#1f77b4', linewidth=2)

# 2. 2023년 예측 데이터 그리기 (22년 4분기 마지막 점과 연결)
# 과거 데이터의 마지막 시점과 매출액 추출
last_actual_time = df_coffee['기준년분기'].iloc[-1]
last_actual_value = df_coffee['총매출'].iloc[-1]

# 과거 마지막 점 + 2023년 4개 분기 예측 데이터 이어붙이기
forecast_x = [last_actual_time] + real_forecast['기준년분기'].tolist()
forecast_y = [last_actual_value] + real_forecast['2023년_예측_총매출'].tolist()

# 예측 구간은 점선(--)과 X 마커로 직관적 표시
plt.plot(forecast_x, forecast_y, marker='X', linestyle='--', label='2023년 미래 예측', color='#ff7f0e', linewidth=2)

# 그래프 레이아웃 및 디자인 다듬기
plt.title('신촌동 커피-음료 업종 총매출 추이 및 2023년 최종 예측 (Prophet)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('기준년분기', fontsize=12)
plt.ylabel('총매출 금액 (원)', fontsize=12)
plt.xticks(rotation=45) # X축 분기 텍스트가 겹치지 않도록 45도 회전
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11, loc='upper left')

# Y축 금액 축 단위 쉼표(,) 표현 조절
current_values = plt.gca().get_yticks()
plt.gca().set_yticklabels(['{:,.0f}'.format(x) for x in current_values])

plt.tight_layout()
plt.show()

## 4. 서울시 상권 모델링 적용

서울 전 상권(383개)를 대상으로 [커피-음료] 업종 매출 예측을 진행하였습니다.
- **분석 과정:** 전 상권의 오차율을 검증할 수 있도록 함수 자동화를 진행하고, 오차율을 기준으로 진단 분류함
- **분석 결과:** 오차율이 10% 이하인 상권 9개(2.35%)를 발견함
- **해석:** 데이터가 24개 분기로 매우 적었으며, 특징이 많은 상권의 매출 요소를 외부 변수없이 예측하는 것은 한계가 있음 
- **의의:** 평균 오차율 6.78 수준으로 예측 가능성이 높은 안정 상권을 선별함 

In [ ]:
import os, sys
import logging
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)


# 1. 전수조사 대상 업종 고정 및 데이터 사전 전처리
target_sector = '커피-음료'
coffee_df_total = df[df['업종명'] == target_sector].copy()

# 미리 날짜 결합 연산을 해두어 속도를 끌어올립니다
coffee_df_total['기준년분기'] = coffee_df_total['기준_년_코드'].astype(str) + "-" + coffee_df_total['기준_분기_코드'].astype(str)

regions_with_coffee = coffee_df_total['행정동'].unique()

# [수정] 에러를 유발하는 이모지 기호를 일반 텍스트 문자로 전면 교체했습니다
print(f"[INFO] '커피-음료' 업종 분석 대상: 총 {len(regions_with_coffee)}개 행정동")
print(f"[START] 성공한 지역만 골라내는 '초고속 전수 스캔'을 시작합니다. (화면 로그 생략)\n")

success_list = []

# 2. 시스템 가동
for region in regions_with_coffee:
    try:
        # 해당 동네 데이터만 압축 필터링
        region_df = coffee_df_total[coffee_df_total['행정동'] == region]
        
        # 분기별 매출 합산 전처리
        agg_df = region_df.groupby(['기준년분기'])[['주중_매출_금액', '주말_매출_금액']].sum().reset_index()
        agg_df['총매출'] = agg_df['주중_매출_금액'] + agg_df['주말_매출_금액']
        agg_df = agg_df.sort_values('기준년분기').reset_index(drop=True)
        
        total_len = len(agg_df)
        # 데이터가 너무 적어 시계열 학습이 불가능한 동네는 사전에 스킵
        if total_len < 15:
            continue
            
        # Prophet 규격 변환
        df_p = pd.DataFrame()
        df_p['ds'] = pd.to_datetime(agg_df['기준년분기'].str.replace('-', 'Q'))
        df_p['y'] = agg_df['총매출']
        
        # 동적 슬라이싱 (최근 3개 분기를 시험지로)
        split_idx = total_len - 3
        train_p = df_p.iloc[:split_idx]
        test_p = df_p.iloc[split_idx:]
        
        # 코로나 lockdowns 설정
        lockdowns = pd.DataFrame({
          'holiday': 'corona',
          'ds': pd.to_datetime(['2020-03-31', '2020-06-30', '2020-09-30', '2020-12-31',
                                '2021-03-31', '2021-06-30', '2021-09-30', '2021-12-31']),
          'lower_window': 0, 'upper_window': 0
        })
        
        # 문자 출력을 완벽하게 차단하고 백그라운드 연산
        sys.stdout = open(os.devnull, 'w')
        model = Prophet(holidays=lockdowns, yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        model.fit(train_p)
        
        future = model.make_future_dataframe(periods=3, freq='Q')
        forecast = model.predict(future)
        sys.stdout = sys.__stdout__ # 출력 복구
        
        pred_val = forecast['yhat'].iloc[split_idx:].values
        
        # 오차율 계산
        error_rmse = rmse(test_p['y'].values, pred_val)
        mean_sales = test_p['y'].mean()
        
        if mean_sales == 0:
            continue
            
        error_rate = (error_rmse / mean_sales) * 100
        
        # 오차율 등급
        if error_rate <= 10: status = "안정 (예측 매우 잘됨)"
        elif error_rate <= 30: status = "주의 (변동성 중간)"
        else: status = "경고 (변동성 큼)"
            
        # 성공한 데이터만 수집
        success_list.append({
            '행정동': region,
            '업종명': target_sector,
            '오차율(NRMSE)': error_rate,
            '시스템진단': status
        })
        
    except Exception as e:
        sys.stdout = sys.__stdout__
        continue

# 시스템 출력 정상화 복구
sys.stdout = sys.__stdout__

# 3. 오차율이 가장 낮은 순서대로 정렬하여 화면에 출력
if len(success_list) > 0:
    result_df = pd.DataFrame(success_list)
    result_df = result_df.sort_values(by='오차율(NRMSE)').reset_index(drop=True)
    result_df['오차율(NRMSE)'] = result_df['오차율(NRMSE)'].apply(lambda x: f"{x:.2f}%")
    
    print("\n" + "="*70)
    print(f"[RESULT] 서울시 [커피-음료] 예측에 성공한 상권 리스트 (오차율 낮은 순)")
    print("="*70)
    display(result_df)
else:
    print("\n[FAIL] 연산에 성공한 지역이 없습니다. 데이터 원본 구조를 재확인해주세요.")

In [ ]:
# =========================================================================
# [커피-음료] 종합 통계 성적표 생성
# =========================================================================

# 1. 등급별 빈도수(개수)와 비율 계산
total_count = len(result_df)
status_counts = result_df['시스템진단'].value_counts()
status_ratios = result_df['시스템진단'].value_counts(normalize=True) * 100

# 2. 등급별 평균 오차율 계산 (문자열 %를 다시 숫자로 바꿔서 계산)
result_df['오차율_숫자'] = result_df['오차율(NRMSE)'].str.replace('%', '').astype(float)
status_means = result_df.groupby('시스템진단')['오차율_숫자'].mean()

# 3. 데이터 합쳐서 예쁜 요약 표 만들기
summary_table = pd.DataFrame({
    '상권 개수(개)': status_counts,
    '비율(%)': status_ratios,
    '그룹별 평균 오차율': status_means
})

# 인덱스 순서 정렬
order = ['안정 (예측 매우 잘됨)', '주의 (변동성 중간)', '경고 (변동성 큼)']
summary_table = summary_table.reindex(order)

# 소수점 둘째 자리 포맷팅
summary_table['비율(%)'] = summary_table['비율(%)'].apply(lambda x: f"{x:.2f}%")
summary_table['그룹별 평균 오차율'] = summary_table['그룹별 평균 오차율'].apply(lambda x: f"{x:.2f}%")

print("="*70)
print(f"📊 서울시 커피-음료 상권 (총 {total_count}개 행정동) 분석 결과 요약")
print("="*70)
display(summary_table)

print("\n" + "="*70)
print("💡 [포트폴리오 프리뷰] 탑플레이스 상권 리스트 (오차율 Top 10)")
print("="*70)
display(result_df[['행정동', '오차율(NRMSE)', '시스템진단']].head(10))